# 01 — Data Pull, Inspect & Validate (Sample Databases)

**Thesis:** Implied Volatility Smile Spillovers (AP-33)  
**Author:** Başar Hacımustafaoğlu — 14866196  
**Purpose:** Pull a small sample from each accessible thesis-relevant database, inspect the structure, and validate the data quality.  

---

## What this notebook does

1. Connects to WRDS
2. For each accessible database: pulls a small sample (top 500 rows)
3. Runs full validation on each pull (shape, dtypes, missingness, date coverage, duplicates)
4. Saves raw pulls to `data/raw/` as CSV
5. Saves a validation summary report to `logs/`

**This notebook does NOT clean, merge, or analyse anything.**  
**It only answers: what does this data look like, and is it usable?**

---

> ⚠️ **Run notebook 00 first** to verify your WRDS connection.

## Step 1 — Imports and connection

In [2]:
import os
import sys
import datetime
import json
import pandas as pd
import wrds
from dotenv import load_dotenv

sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from wrds_utils import connect_wrds, validate_df

load_dotenv()
conn = connect_wrds()

TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RAW_DIR   = "../data/raw"
LOG_DIR   = "../logs"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Run timestamp: {TIMESTAMP}")

Connecting to WRDS as: basar
Loading library list...
Done
Connection established.
Run timestamp: 20260308_164455


## Step 2 — Define pull targets

For each database we have access to, we specify which table to pull a sample from and why.

In [3]:
# Each entry: (library, table, description, n_rows_to_pull)
# We pull only 500 rows per table — this is a structural inspection, not a data download

PULL_TARGETS = [

    # OptionMetrics US sample (2014)
    ("optionmsamp_us",     "vsurfd2014",              "OptionMetrics US — volatility surface (2014)",        500),
    ("optionmsamp_us",     "opprcd2014",              "OptionMetrics US — raw option prices (2014)",         500),
    ("optionmsamp_us",     "secnmd",                  "OptionMetrics US — security names",                   500),
    ("optionmsamp_us",     "secprd",                  "OptionMetrics US — underlying prices",                500),

    # OptionMetrics Europe sample (2013)
    ("optionmsamp_europe", "volatility_surface_2013", "OptionMetrics EU — volatility surface (2013)",        500),
    ("optionmsamp_europe", "option_price_2013",       "OptionMetrics EU — raw option prices (2013)",         500),
    ("optionmsamp_europe", "security_name",           "OptionMetrics EU — security names",                   500),
    ("optionmsamp_europe", "security_price",          "OptionMetrics EU — underlying prices",                500),
    ("optionmsamp_europe", "historical_volatility",   "OptionMetrics EU — historical volatility",            500),

    # CBOE
    ("cboe",               "cboe",                    "CBOE — VIX and volatility indices",                   500),

    # FRB
    ("frb",                "rates_daily",             "FRB — daily interest rates",                          500),

    # CRSP
    ("crsp",               "dsf",                     "CRSP — daily stock file",                             500),
    ("crsp",               "dsp500",                  "CRSP — daily S&P 500 index",                          500),
    ("crsp",               "dsi",                     "CRSP — daily market index",                           500),

    # Compustat Global
    ("comp",               "g_secd",                  "Compustat Global — daily security prices",            500),
    ("comp",               "g_idx_daily",             "Compustat Global — daily index prices",               500),
    ("comp",               "g_exrt_dly",              "Compustat Global — daily exchange rates",             500),

    # Fama-French
    ("ff",                 "factors_daily",           "Fama-French — daily factors",                         500),


    # WRDS Apps
    ("wrdsapps",           "eushort",                 "WRDS Apps — EU short sales",                          500),
    ("wrdsapps",           "intl_market_returns",     "WRDS Apps — international market returns",            500),

    # MacroFin
    ("macrofin",           "q_factors_daily",         "MacroFin — Q-factors daily",                          500),
]

print(f"Pull targets defined: {len(PULL_TARGETS)} tables")

Pull targets defined: 24 tables


## Step 3 — Pull and validate each table

For each target we:
1. Pull `n` rows
2. Run validation
3. Save raw CSV to `data/raw/`
4. Log the result

Errors are caught and logged — a failed pull does not stop the notebook.

In [12]:
# ================================================================
# STEP 3 — PULL DATA WITH DATE FILTER (MODERN PERIOD)
# Pulls 500 most recent rows per table after 2010-01-01
# For inspection only — full pull happens in notebook 02
# ================================================================

pull_log   = []
pulled_dfs = {}

# Tables that have a date column — we filter to modern period
DATE_FILTERED = {
    "optionmsamp_us.vsurfd2014":                  "date",
    "optionmsamp_us.opprcd2014":                  "date",
    "optionmsamp_us.secprd":                      "date",
    "optionmsamp_europe.volatility_surface_2013": "date",
    "optionmsamp_europe.option_price_2013":       "date",
    "optionmsamp_europe.security_price":          "date",
    "optionmsamp_europe.historical_volatility":   "date",
    "cboe.cboe":                                  "date",
    "frb.rates_daily":                            "date",
    "crsp.dsi":                                   "date",
    "crsp.dsf":                                   "date",
    "crsp.dsp500":                                "caldt", 
    "comp.g_secd":                                "datadate", 
    "comp.g_idx_daily":                           "datadate", 
    "comp.g_exrt_dly":                            "datadate", 
    "ff.factors_daily":                           "date",
    "djones.djdaily":                             "date",
    "phlx.iv":                                    "qdate",
    "wrdsapps.eushort":                           "position_date",
    "macrofin.q_factors_daily":                   "date",
}

for library, table, description, n_rows in PULL_TARGETS:
    key = f"{library}.{table}"

    print(f"\n{'='*60}")
    print(f"Pulling: {key}")
    print(f"{'='*60}")

    log_entry = {
        "library":     library,
        "table":       table,
        "description": description,
        "n_requested": n_rows,
        "status":      None,
        "n_rows":      None,
        "n_cols":      None,
        "columns":     None,
        "error":       None,
        "saved_to":    None,
    }

    try:
        if key in DATE_FILTERED:
            date_col = DATE_FILTERED[key]
            if key == "comp.g_secd":
                sql = f"""
                    SELECT * FROM {library}.{table}
                    WHERE {date_col} >= '2010-01-01'
                    AND {date_col} <= '2024-12-31'
                    AND prccd IS NOT NULL
                    ORDER BY {date_col} DESC
                    LIMIT {n_rows}
                """
            else:
                sql = f"""
                    SELECT * FROM {library}.{table}
                    WHERE {date_col} >= '2010-01-01'
                    ORDER BY {date_col} DESC
                    LIMIT {n_rows}
                """
        else:
            sql = f"""
                SELECT * FROM {library}.{table}
                LIMIT {n_rows}
            """

        df = conn.raw_sql(sql)

        log_entry["status"]  = "SUCCESS"
        log_entry["n_rows"]  = len(df)
        log_entry["n_cols"]  = len(df.columns)
        log_entry["columns"] = list(df.columns)

        validate_df(df, key)

        filename = f"{library}__{table}__MODERN_{TIMESTAMP}.csv"
        filepath = os.path.join(RAW_DIR, filename)
        df.to_csv(filepath, index=False)
        log_entry["saved_to"] = filepath
        print(f"✓ Saved to: {filepath}")

        pulled_dfs[key] = df

    except Exception as e:
        log_entry["status"] = "ERROR"
        log_entry["error"]  = str(e)
        print(f"✗ ERROR: {e}")
        print("  Skipping and continuing.")

    pull_log.append(log_entry)

print(f"\n{'='*60}")
print("All pull attempts complete.")


Pulling: optionmsamp_us.vsurfd2014

VALIDATION REPORT: optionmsamp_us.vsurfd2014

[1] Shape: 500 rows x 9 columns

[2] Columns and dtypes:
    secid                               Float64
    date                                string
    days                                Float64
    delta                               Float64
    impl_volatility                     Float64
    impl_strike                         Float64
    impl_premium                        Float64
    dispersion                          Float64
    cp_flag                             string

[3] Missingness:
    No missing values detected.

[4] Date coverage:
    date: 2014-03-13 00:00:00 → 2014-03-14 00:00:00

[5] Duplicate rows: 0


✓ Saved to: ../data/raw/optionmsamp_us__vsurfd2014__MODERN_20260308_164455.csv

Pulling: optionmsamp_us.opprcd2014

VALIDATION REPORT: optionmsamp_us.opprcd2014

[1] Shape: 500 rows x 22 columns

[2] Columns and dtypes:
    secid                               Float64
    date       

In [7]:
# Quick row name check for pull date fix
df = conn.raw_sql("""
    SELECT gvkey, datadate, conm, prccd, curcdd, loc
    FROM comp.g_secd
    WHERE datadate BETWEEN '2010-01-01' AND '2024-12-31'
    AND prccd IS NOT NULL
    ORDER BY datadate DESC
    LIMIT 10
""")
print(df.to_string())

    gvkey    datadate                          conm    prccd curcdd  loc
0  371311  2024-12-31  INTERCAPITAL KROVNI UCITS ET    10.15    EUR  HRV
1  371169  2024-12-31  BANDHAN MUTUAL FUND - BANDHA   257.03    INR  IND
2  371125  2024-12-31            NOBLE POLYMERS LTD     0.31    INR  IND
3  371062  2024-12-31     IPA SECURITIES INVESTMENT   8500.0    VND  VNM
4  371026  2024-12-31   KSM MUTUAL FUNDS LTD. - KSM  42.2507    ILS  ISR
5  371025  2024-12-31   MEITAV TACHLIT MUTUAL FUNDS    4.719    ILS  ISR
6  371014  2024-12-31    KSM MUTUAL FUNDS LTD - KSM    82.99    ILS  ISR
7  371013  2024-12-31   KSM MUTUAL FUNDS LTD. - KSM  41.7482    ILS  ISR
8  371012  2024-12-31      INMCAMSAL GESTION SIL SA   1.0038    EUR  ESP
9  371011  2024-12-31       CHINA ASSET MGMT CO LTD     1.06    CNY  CHN


## Step 4 — Pull summary

## Step 5 — Interactive inspection

Use the cells below to look at specific tables in more detail.  
The most important ones for your thesis are the volatility surface tables.

## Step 6 — Save full log

## Step 7 — Close connection